# Module 00 — Lab: Your first Claude request

Goals:
1. Confirm your environment is wired up.
2. Send a single request and read every field of the response.
3. See multi-turn in action.
4. Switch models and compare.

Prereqs: `pip install -r ../requirements.txt` and a populated `.env`.

## 1. Setup

In [ ]:
import os
import time
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(dotenv_path='../.env')
assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY in .env'

client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')
FAST  = os.getenv('ANTHROPIC_FAST_MODEL', 'claude-haiku-4-5-20251001')
print('default model:', MODEL)

## 2. First request

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system='You answer in one short sentence.',
    messages=[{'role': 'user', 'content': 'What is prompt engineering?'}],
)

print(response.content[0].text)
print(f"\n[stop={response.stop_reason}  in={response.usage.input_tokens}  out={response.usage.output_tokens}]")

## 3. Inspect the full response

Look at every field. Notice `content` is a *list* of blocks, not a string.

In [ ]:
print(response.model_dump_json(indent=2))

## 4. Multi-turn in one call

The model is stateless. You pass the whole history every time.

In [ ]:
messages = [
    {'role': 'user',      'content': 'Pick a number between 1 and 10. Reply with only the number.'},
    {'role': 'assistant', 'content': '7'},
    {'role': 'user',      'content': 'Now double it. Reply with only the result.'},
]
r = client.messages.create(model=MODEL, max_tokens=16, messages=messages)
print(r.content[0].text)

## 5. Compare Sonnet vs Haiku

Same prompt, two models. Look at latency, output, and token usage.

In [ ]:
PROMPT = 'In 30 words, explain what an LLM is to a senior software engineer.'

def time_request(model):
    t0 = time.perf_counter()
    r = client.messages.create(
        model=model, max_tokens=128,
        messages=[{'role': 'user', 'content': PROMPT}],
    )
    dt = time.perf_counter() - t0
    return dt, r

for m in (MODEL, FAST):
    dt, r = time_request(m)
    print(f'\n--- {m}  ({dt*1000:.0f} ms) ---')
    print(r.content[0].text)
    print(f'in={r.usage.input_tokens}  out={r.usage.output_tokens}')

## 6. Stop reasons

Cap `max_tokens` aggressively and watch `stop_reason` flip to `max_tokens`.

In [ ]:
r = client.messages.create(
    model=MODEL, max_tokens=8,
    messages=[{'role': 'user', 'content': 'Write a 200-word essay about coffee.'}],
)
print('text:', r.content[0].text)
print('stop_reason:', r.stop_reason)

## 7. Count tokens *without* a request

Useful for estimating cost / fitting context before you spend money.

In [ ]:
r = client.messages.count_tokens(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'the quick brown fox jumps over the lazy dog'}],
)
print('input_tokens:', r.input_tokens)

---

### What you should now feel comfortable with

- The shape of a request and response.
- That `content` is a typed list of blocks.
- That you control history; the model doesn't remember.
- That swapping models is a one-line change.

Next: **Module 01 — Prompting.**